In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
ROOT.gStyle.SetOptStat(0)

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
figPath = dirDir + "figures/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
filename = dataPath + "DmlNtup_Data_9years_FullPhotonData_AllSky.root"

# Pass the list of files to RDataFrame
rdf = ROOT.RDataFrame("DmlNtup", filename)

In [ ]:
Emin = 0.1
Emax = 1e4
n_E_bins = 50 # 40

In [ ]:

def create_log_bins(min_val, max_val, n_bins):
    """
    Return logarithmically spaced bin edges.
    """
    return np.logspace(np.log10(min_val), np.log10(max_val), n_bins + 1)

In [ ]:
log_bins = create_log_bins(Emin, Emax, n_E_bins)
log_bins_root = ROOT.std.vector('double')(log_bins.tolist())

### Study how LET \ HET evolve over energy (and angle ?)

In [ ]:

h = rdf.Histo1D(("spectrum", "Spectrum of accepted data;Energy [GeV];Counts", n_E_bins, log_bins_root.data()), "tt_bgoTotalEcorr_GeV")

In [ ]:
def declare_fit_function():
    cpp_code = f"""
        #include <cmath>        
        double powerLaw(double *x, double a, double b){{
            return b*pow(x[0], a);
        }}
        double fitFunction(double *x, double *par){{
            return powerLaw(x, par[0], par[1]);
        }}
        """
        
    ROOT.gInterpreter.Declare(cpp_code)

In [ ]:
declare_fit_function()
# perform the fit
EminFit = 10

fitFuncE = ROOT.TF1(f"fitFunction", ROOT.fitFunction, xmin=EminFit, xmax=Emax, npar=2)
fitFuncE.SetParameter(0, -2.5) # a
fitFuncE.SetParameter(1, 4)    # b 
fitFuncE.SetLineColor(ROOT.kBlack)
fitFuncE.SetLineWidth(2)

# Fit with "R+" option to draw the fit function
res = h.Fit(fitFuncE, "R+")

In [ ]:
c = ROOT.TCanvas()
h.Draw()
alpha = fitFuncE.GetParameter(0)
beta = np.log10(fitFuncE.GetParameter(1))
leg = ROOT.TLegend(0.55, 0.7, 0.82, 0.8)
leg.AddEntry(fitFuncE, f"log_{{10}}(y) \propto {alpha:.2f} log_{{10}}(x)", "l")
leg.SetTextSize(0.03)
leg.Draw("same")
c.SetLogy()
c.SetLogx()
c.SaveAs(figPath+"explo/spectrumAccepted.pdf")
c.Draw()